In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from skmultilearn.adapt import MLkNN
from scipy.sparse import csr_matrix
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_selection import SelectKBest, chi2
import matplotlib.pyplot as plt
import seaborn as sns
import re
from nltk.stem import WordNetLemmatizer

#Carga de datos
train_data = pd.read_csv("../../Data/train_indexado.csv")
test_data = pd.read_csv("../../Data/test_indexado.csv")

# Definir las clases de emociones
emotion_classes = train_data.columns[2:].tolist()

# --- PREPROCESAMIENTO PARA LEMATIZACIÓN ---
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    words = re.findall(r'\b\w+\b', text.lower())
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(lemmatized_words)

X_train_lem = train_data['Text'].apply(preprocess_text)
X_test_lem = test_data['Text'].apply(preprocess_text)

# TF-IDF VECTORIZACIÓN
vectorizer = TfidfVectorizer(lowercase=True, strip_accents="unicode", max_features=10000)
X_train = vectorizer.fit_transform(X_train_lem)
X_test = vectorizer.transform(X_test_lem)
y_train = np.asarray(train_data[emotion_classes])
y_test = np.asarray(test_data[emotion_classes])

In [ ]:
# Almacenar resultados para plotting
results = {
    'k_values': [],
    'SVM_accuracy': [], 'SVM_f1': [], 'SVM_recall': [],
    'RF_accuracy': [], 'RF_f1': [], 'RF_recall': [],
    'MLP_accuracy': [], 'MLP_f1': [], 'MLP_recall': [],
    'MLkNN_accuracy': [], 'MLkNN_f1': [], 'MLkNN_recall': []
}

for k in [500, 1000, 2000, 3000, 5000]:
    print(f"\n*** Evaluación con Chi-cuadrada (Número de atributos: {k}) ***")
    results['k_values'].append(k)

    # Selección de features
    selector = SelectKBest(score_func=chi2, k=k)
    X_train_chi = selector.fit_transform(X_train, y_train)
    X_test_chi = selector.transform(X_test)

    # SVM
    svm_clf = SVC(kernel='linear')
    multi_svm = MultiOutputClassifier(svm_clf)
    multi_svm.fit(X_train_chi, y_train)
    y_pred_svm = multi_svm.predict(X_test_chi)
    
    report_svm = classification_report(y_test, y_pred_svm, output_dict=True, zero_division=0)
    results['SVM_accuracy'].append(accuracy_score(y_test, y_pred_svm))
    results['SVM_f1'].append(report_svm["macro avg"]["f1-score"])
    results['SVM_recall'].append(report_svm["macro avg"]["recall"])
    
    print(f"SVM - Accuracy: {results['SVM_accuracy'][-1]:.5f}, F1: {results['SVM_f1'][-1]:.5f}, Recall: {results['SVM_recall'][-1]:.5f}")

    # Random Forest
    rf_clf = RandomForestClassifier(random_state=42)
    multi_rf = MultiOutputClassifier(rf_clf)
    multi_rf.fit(X_train_chi, y_train)
    y_pred_rf = multi_rf.predict(X_test_chi)
    
    report_rf = classification_report(y_test, y_pred_rf, output_dict=True, zero_division=0)
    results['RF_accuracy'].append(accuracy_score(y_test, y_pred_rf))
    results['RF_f1'].append(report_rf["macro avg"]["f1-score"])
    results['RF_recall'].append(report_rf["macro avg"]["recall"])
    
    print(f"RF - Accuracy: {results['RF_accuracy'][-1]:.5f}, F1: {results['RF_f1'][-1]:.5f}, Recall: {results['RF_recall'][-1]:.5f}")

    # Neural Network
    mlp_clf = MLPClassifier(max_iter=1000)
    multi_mlp = MultiOutputClassifier(mlp_clf)
    multi_mlp.fit(X_train_chi, y_train)
    y_pred_mlp = multi_mlp.predict(X_test_chi)
    
    report_mlp = classification_report(y_test, y_pred_mlp, output_dict=True, zero_division=0)
    results['MLP_accuracy'].append(accuracy_score(y_test, y_pred_mlp))
    results['MLP_f1'].append(report_mlp["macro avg"]["f1-score"])
    results['MLP_recall'].append(report_mlp["macro avg"]["recall"])
    
    print(f"MLP - Accuracy: {results['MLP_accuracy'][-1]:.5f}, F1: {results['MLP_f1'][-1]:.5f}, Recall: {results['MLP_recall'][-1]:.5f}")

    # MLkNN
    mlknn = MLkNN(k=3)
    mlknn.fit(X_train_chi, csr_matrix(y_train))
    y_pred_mlknn = mlknn.predict(X_test_chi)
    
    report_mlknn = classification_report(y_test, y_pred_mlknn, output_dict=True, zero_division=0)
    results['MLkNN_accuracy'].append(accuracy_score(y_test, y_pred_mlknn))
    results['MLkNN_f1'].append(report_mlknn["macro avg"]["f1-score"])
    results['MLkNN_recall'].append(report_mlknn["macro avg"]["recall"])
    
    print(f"MLkNN - Accuracy: {results['MLkNN_accuracy'][-1]:.5f}, F1: {results['MLkNN_f1'][-1]:.5f}, Recall: {results['MLkNN_recall'][-1]:.5f}")


*** Evaluación con Chi-cuadrada (Número de atributos: 500) ***
SVM - Accuracy: 0.11369, F1: 0.31468, Recall: 0.70171
RF - Accuracy: 0.32615, F1: 0.32095, Recall: 0.25938
MLP - Accuracy: 0.33849, F1: 0.34945, Recall: 0.29959
MLkNN - Accuracy: 0.33002, F1: 0.30450, Recall: 0.23714

*** Evaluación con Chi-cuadrada (Número de atributos: 1000) ***
SVM - Accuracy: 0.10540, F1: 0.32508, Recall: 0.70944
RF - Accuracy: 0.31620, F1: 0.32013, Recall: 0.25229
MLP - Accuracy: 0.32523, F1: 0.36687, Recall: 0.31568
MLkNN - Accuracy: 0.32136, F1: 0.26145, Recall: 0.21955

*** Evaluación con Chi-cuadrada (Número de atributos: 2000) ***
SVM - Accuracy: 0.09416, F1: 0.34401, Recall: 0.68853
RF - Accuracy: 0.30256, F1: 0.30897, Recall: 0.24110
MLP - Accuracy: 0.32043, F1: 0.35505, Recall: 0.30932
MLkNN - Accuracy: 0.22370, F1: 0.23437, Recall: 0.19196

*** Evaluación con Chi-cuadrada (Número de atributos: 3000) ***
SVM - Accuracy: 0.09434, F1: 0.35441, Recall: 0.66935
RF - Accuracy: 0.30311, F1: 0.29597,

In [ ]:
# Crear plots comparativos
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plot 1: Accuracy
axes[0].plot(results['k_values'], results['SVM_accuracy'], 'o-', label='SVM', linewidth=2, markersize=8)
axes[0].plot(results['k_values'], results['RF_accuracy'], 's-', label='Random Forest', linewidth=2, markersize=8)
axes[0].plot(results['k_values'], results['MLP_accuracy'], '^-', label='Neural Network', linewidth=2, markersize=8)
axes[0].plot(results['k_values'], results['MLkNN_accuracy'], 'd-', label='MLkNN', linewidth=2, markersize=8)
axes[0].set_xlabel('Número de Features (Chi2)', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Comparación de Accuracy por Número de Features', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: F1 Score
axes[1].plot(results['k_values'], results['SVM_f1'], 'o-', label='SVM', linewidth=2, markersize=8)
axes[1].plot(results['k_values'], results['RF_f1'], 's-', label='Random Forest', linewidth=2, markersize=8)
axes[1].plot(results['k_values'], results['MLP_f1'], '^-', label='Neural Network', linewidth=2, markersize=8)
axes[1].plot(results['k_values'], results['MLkNN_f1'], 'd-', label='MLkNN', linewidth=2, markersize=8)
axes[1].set_xlabel('Número de Features (Chi2)', fontsize=12)
axes[1].set_ylabel('F1 Score (Macro)', fontsize=12)
axes[1].set_title('Comparación de F1 Score por Número de Features', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Recall
axes[2].plot(results['k_values'], results['SVM_recall'], 'o-', label='SVM', linewidth=2, markersize=8)
axes[2].plot(results['k_values'], results['RF_recall'], 's-', label='Random Forest', linewidth=2, markersize=8)
axes[2].plot(results['k_values'], results['MLP_recall'], '^-', label='Neural Network', linewidth=2, markersize=8)
axes[2].plot(results['k_values'], results['MLkNN_recall'], 'd-', label='MLkNN', linewidth=2, markersize=8)
axes[2].set_xlabel('Número de Features (Chi2)', fontsize=12)
axes[2].set_ylabel('Recall (Macro)', fontsize=12)
axes[2].set_title('Comparación de Recall por Número de Features', fontsize=14)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../../Plots/chi2_models_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# Crear tabla resumen
df_results = pd.DataFrame(results)
print("\n=== TABLA RESUMEN DE RESULTADOS ===")
print(df_results.round(5))